<a href="https://colab.research.google.com/github/aishanikar9/BWSI_Operations_Team/blob/main/BWSI_xView2_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import pandas as pd
from skimage import io
import numpy as np
import matplotlib.pyplot as plt
import pathlib


In [2]:
import torch
import gc

# Flush all residual variables and cache from previous execution states
gc.collect()
torch.cuda.empty_cache()
print("GPU memory cleared.")


GPU memory cleared.


In [ ]:
!pip install kagglehub

import kagglehub

# Download latest version
path = kagglehub.dataset_download("tunguz/xview2-challenge-dataset-train-and-test")
print("Path to dataset files:", path)

import os

# View all files and folders in the downloaded path
files = os.listdir(path)
print(files)


 77%|███████▋  | 7.95G/10.3G [01:31<00:16, 156MB/s]

In [ ]:
# To inspect if folders are structures or archives, add this block:
for f in files:
    full_p = os.path.join(path, f)
    if os.path.isdir(full_p):
        print(f"Folder: {f} -> Contains: {os.listdir(full_p)}")
    else:
        print(f"File: {f}")


In [ ]:
# Check the contents inside the nested folders
print("Inside train/train:", os.listdir(os.path.join(path, 'train', 'train')))
print("Inside test/test:", os.listdir(os.path.join(path, 'test', 'test')))


In [ ]:
!pip install shapely

first we find the bounding box of each building and crop them out into separate chips to make our new target dataset.

In [ ]:
import os
import json
import cv2
import pandas as pd
import numpy as np
import kagglehub
from shapely.wkt import loads
from sklearn.model_selection import train_test_split
from tqdm import tqdm

path = kagglehub.dataset_download("tunguz/xview2-challenge-dataset-train-and-test")

train_dir = os.path.join(path, 'train', 'train')
train_image_dir = os.path.join(train_dir, 'images')
train_label_dir = os.path.join(train_dir, 'labels')

test_dir = os.path.join(path, 'test', 'test')
test_image_dir = os.path.join(test_dir, 'images')

output_crop_dir = '/content/cropped_building_chips'
os.makedirs(output_crop_dir, exist_ok=True)

damage_map = {'no-damage': 0, 'minor-damage': 1, 'major-damage': 2, 'destroyed': 3}
post_jsons = [f for f in os.listdir(train_label_dir) if f.endswith('_post_disaster.json')]

all_train_val_chips = []
crop_counter = 0

print("--- Step 1: Extracting Building Chips from Training Images ---")
for j_file in tqdm(post_jsons):
    img_file = j_file.replace('.json', '.png')
    img_path = os.path.join(train_image_dir, img_file)

    if not os.path.exists(img_path):
        continue

    img = cv2.imread(img_path)
    if img is None:
        continue

    with open(os.path.join(train_label_dir, j_file)) as f:
        label_data = json.load(f)

    for feature in label_data['features']['xy']:
        wkt_string = feature['wkt']
        damage_type = feature['properties']['subtype']

        if damage_type not in damage_map:
            continue

        try:
            poly = loads(wkt_string)
            xmin, ymin, xmax, ymax = poly.bounds

            xmin, ymin = max(0, int(xmin)), max(0, int(ymin))
            xmax, ymax = min(img.shape[1], int(xmax)), min(img.shape[0], int(ymax))

            crop = img[ymin:ymax, xmin:xmax]
            if crop.size == 0 or crop.shape[0] < 10 or crop.shape[1] < 10:
                continue

            crop_filename = f"building_{crop_counter}.png"
            crop_path = os.path.join(output_crop_dir, crop_filename)
            cv2.imwrite(crop_path, crop)

            all_train_val_chips.append({
                'chip_path': crop_path,
                'damage_label': damage_map[damage_type]
            })
            crop_counter += 1

        except Exception:
            continue

print(f"\nExtracted {crop_counter} building chips.")

print("\n--- Step 2: Creating xView2 Train and Val CSV Catalogs ---")
df_all = pd.DataFrame(all_train_val_chips)

df_train, df_val = train_test_split(df_all, test_size=0.2, stratify=df_all['damage_label'], random_state=42)

df_train.to_csv('/content/xview2-classification-train.csv', index=False)
df_val.to_csv('/content/xview2-classification-val.csv', index=False)

print(f"Saved: xview2-classification-train.csv ({len(df_train)} records)")
print(f"Saved: xview2-classification-val.csv ({len(df_val)} records)")

print("\n--- Step 3: Cataloging Unlabeled xView2 Test Images ---")
test_images = [os.path.join(test_image_dir, f) for f in os.listdir(test_image_dir) if f.endswith('.png')]

df_test = pd.DataFrame({'image_path': test_images})
df_test.to_csv('/content/xview2-classification-test.csv', index=False)
print(f"Saved: xview2-classification-test.csv ({len(df_test)} raw testing images)")


In [ ]:
# from google.colab import files

# files.download('/content/xview2-classification-train.csv')
# files.download('/content/xview2-classification-val.csv')
# files.download('/content/xview2-classification-test.csv')

In [ ]:
# #weighting each class

import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision

train_df = pd.read_csv('/content/xview2-classification-train.csv')

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Active training device: {device}")

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision

#Compute Damped Balanced Weights
count = train_df["damage_label"].value_counts().sort_index()
class_weights = torch.tensor((count.sum() / (len(count) * count)).values, dtype=torch.float).to(device)
print("Balanced Class Weights calculated:", class_weights)

net = torchvision.models.resnet50(weights='IMAGENET1K_V2')
net.fc = nn.Sequential( #dropout and output linear layer
    nn.Dropout(p=0.3),
    nn.Linear(2048, 4)
)
net = net.to(device)

for parameters in net.parameters():
    parameters.requires_grad = False

for parameters in net.fc.parameters():
    parameters.requires_grad = True

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.Adam(filter(lambda p: p.requires_grad, net.parameters()), lr=1e-3)
scheduler = optim.lr_scheduler.ExponentialLR(optimizer, gamma=0.9)

In [ ]:
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from PIL import Image
import torchvision.transforms as transforms
import pandas as pd
import torch

# Define Individual Transformation Components at 224x224
flip = transforms.RandomHorizontalFlip(p=0.5)
v_flip = transforms.RandomVerticalFlip(p=0.5)
scale = transforms.Resize((224, 224))
rotation = transforms.RandomRotation(degrees=30)
jitter = transforms.ColorJitter(brightness=0.25, contrast=0.25, saturation=0.15)
perspective = transforms.RandomPerspective(distortion_scale=0.2, p=0.4)

# Compose the Pipelines
composed_train = transforms.Compose([
    scale, flip, v_flip, rotation, perspective, jitter,
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

composed_val = transforms.Compose([
    scale,
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Custom Dataset Class
class XView2Dataset(Dataset):
    def __init__(self, label_csv, transform=None):
        self.label_data_df = pd.read_csv(label_csv)
        self.transform = transform

    def __len__(self):
        return len(self.label_data_df)

    def __getitem__(self, idx):
        if torch.is_tensor(idx):
            idx = idx.tolist()
        img_path = self.label_data_df.iloc[idx]['chip_path']
        label = int(self.label_data_df.iloc[idx]['damage_label'])
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return {'image': image, 'label': label}

# Instantiate Datasets
transformed_train_dataset = XView2Dataset('/content/xview2-classification-train.csv', transform=composed_train)
transformed_val_dataset = XView2Dataset('/content/xview2-classification-val.csv', transform=composed_val)

# Compute Class Weights for the WeightedRandomSampler
train_df = pd.read_csv('/content/xview2-classification-train.csv')
class_counts = train_df['damage_label'].value_counts().sort_index().values
class_weights = 1.0 / class_counts
sample_weights = [class_weights[label] for label in train_df['damage_label']]

# Setup Sampler
sampler = WeightedRandomSampler(weights=sample_weights, num_samples=len(sample_weights), replacement=True)

# Set Up Parallel DataLoaders (shuffle=False is strictly enforced here to prevent crashing)
train_loader = DataLoader(transformed_train_dataset, batch_size=32, sampler=sampler, num_workers=2, pin_memory=True)
val_loader = DataLoader(transformed_val_dataset, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)


In [ ]:
import pathlib
import copy
import torch
from torch.utils.tensorboard import SummaryWriter
from sklearn.metrics import f1_score
import numpy as np

def train_model(net, train_loader, val_loader, criterion, optimizer, scheduler, logs_path, model_name,
                starting_epoch=0, additional_epochs=30, print_every_num_batches=100, patience=3):
    model_name_base = f'resnet50-{model_name}' + '.ep{}.pth'
    writer = SummaryWriter(logs_path)
    checkpoints_path = logs_path / 'checkpoints'
    checkpoints_path.mkdir(parents=True, exist_ok=True)

    if starting_epoch > 0:
        starting_epoch_string = str(starting_epoch).zfill(3)
        model_load_path = checkpoints_path / model_name_base.format(starting_epoch_string)
        net.load_state_dict(torch.load(model_load_path))

    use_amp = torch.cuda.is_available()
    scaler = torch.amp.GradScaler('cuda', enabled=use_amp)

    best_val_acc = 0.0

    history = {
        'train_loss': [], 'val_loss': [],
        'train_acc': [], 'val_acc': [], 'val_macro_f1': []
    }

    for epoch in range(starting_epoch, starting_epoch + additional_epochs):
        #Training Phase
        net.train()
        running_loss, running_epoch_loss = 0.0, 0.0
        train_correct, train_total = 0, 0

        for i, data in enumerate(train_loader, 0):
            inputs = data['image'].to(device)
            labels = data['label'].to(device)

            optimizer.zero_grad()
            with torch.amp.autocast(device_type='cuda' if use_amp else 'cpu', enabled=use_amp):
                outputs = net(inputs)
                loss = criterion(outputs, labels)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            running_loss += loss.item()
            running_epoch_loss += loss.item()

            predicted = outputs.argmax(dim=1)
            train_total += labels.size(0)
            train_correct += (predicted == labels).sum().item()

            if (i + 1) % print_every_num_batches == 0:
                print(f'[epoch {epoch + 1}, batch {i + 1}] average loss: {running_loss / print_every_num_batches:.4f}')
                running_loss = 0.0

        average_epoch_loss = running_epoch_loss / (i + 1)
        epoch_train_accuracy = 100.0 * train_correct / train_total
        writer.add_scalar('Loss/epoch_avg/train', average_epoch_loss, epoch)
        print(f'[epoch {epoch + 1}] average training epoch loss: {average_epoch_loss:.4f}')
        print(f'[epoch {epoch + 1}] training accuracy: {epoch_train_accuracy:.2f}%')

        writer.add_scalar('LR/rate', scheduler.get_last_lr()[0], epoch)
        scheduler.step()

        #Validation Phase
        net.eval()
        running_val_loss = 0.0
        val_correct, val_total = 0, 0
        val_preds_epoch, val_labels_epoch = [], []

        print("Getting epoch validation loss...")
        with torch.no_grad():
            for i, data in enumerate(val_loader, 0):
                inputs = data['image'].to(device)
                labels = data['label'].to(device)

                with torch.amp.autocast(device_type='cuda' if use_amp else 'cpu', enabled=use_amp):
                    outputs = net(inputs)
                    loss = criterion(outputs, labels)

                running_val_loss += loss.item()
                predicted = outputs.argmax(dim=1)
                val_total += labels.size(0)
                val_correct += (predicted == labels).sum().item()

                val_preds_epoch.extend(predicted.cpu().numpy())
                val_labels_epoch.extend(labels.cpu().numpy())

        average_val_loss = running_val_loss / (i + 1)
        val_accuracy = 100.0 * val_correct / val_total
        epoch_macro_f1 = f1_score(np.array(val_labels_epoch), np.array(val_preds_epoch), average='macro', zero_division=0)

        writer.add_scalar('Loss/epoch_avg/val', average_val_loss, epoch)
        writer.add_scalar('Accuracy/epoch_avg/val', val_accuracy, epoch)
        print(f'[epoch {epoch + 1}] average val epoch loss: {average_val_loss:.4f}')
        print(f'[epoch {epoch + 1}] val accuracy: {val_accuracy:.2f}%')

        history['train_loss'].append(average_epoch_loss)
        history['val_loss'].append(average_val_loss)
        history['train_acc'].append(epoch_train_accuracy)
        history['val_acc'].append(val_accuracy)
        history['val_macro_f1'].append(epoch_macro_f1)

        epoch_string = str(epoch + 1).zfill(3)
        model_save_path = checkpoints_path / model_name_base.format(epoch_string)
        torch.save(net.state_dict(), model_save_path)

        if val_accuracy > best_val_acc:
            best_val_acc = val_accuracy
            best_model_path = checkpoints_path / f"resnet50-{model_name}_BEST.pth"
            torch.save(net.state_dict(), best_model_path)
            print(f"--> New optimal accuracy achieved ({best_val_acc:.2f}%). Standalone BEST file overwritten.")
        else:
            print(f"--> Accuracy did not improve. Current best remains: {best_val_acc:.2f}%")

    print('Finished Training Pipeline Execution Window')
    writer.close()
    return history


first 5 epochs with the updated stuff

In [ ]:
import pathlib

outputs = pathlib.Path('outputs')
outputs.mkdir(exist_ok=True, parents=True)
model_name = 'damage_model'

#Establish fresh model structure with layer freezing
net = torchvision.models.resnet50(weights='IMAGENET1K_V2')
net.fc = nn.Sequential(
    nn.Dropout(p=0.3),
    nn.Linear(2048, 4)
)
net = net.to(device)

for parameters in net.parameters():
    parameters.requires_grad = False
for parameters in net.fc.parameters():
    parameters.requires_grad = True

train_model(net, train_loader, val_loader, criterion, optimizer, scheduler, outputs, model_name,
            starting_epoch=0, additional_epochs=5, print_every_num_batches=100)


epochs 6 to 36 here:

In [ ]:
#Unfreeze the entire backbone network structure
for parameters in net.parameters():
    parameters.requires_grad = True

#Lower learning rate so it safely fine-tunes
optimizer = optim.Adam(net.parameters(), lr=1e-4)
scheduler = optim.lr_scheduler.ExponentialLR(optimizer, gamma=0.95)

print("Backbone layers fully unfrozen. Extended 10-epoch patience limit active.")

training_history = train_model(net, train_loader, val_loader, criterion, optimizer, scheduler, outputs, model_name,
                               starting_epoch=5, additional_epochs=25, print_every_num_batches=100, patience=10)

plot_training_history(training_history)


In [ ]:
# Compress your checkpoints folder into a single file and download it instantly
!zip -r xview2_checkpoints.zip outputs/checkpoints/
from google.colab import files
files.download('xview2_checkpoints.zip')

training history curves

In [ ]:
import matplotlib.pyplot as plt

def plot_training_history(history_dict):
    epochs = range(1, len(history_dict['train_loss']) + 1)

    #Plot Training and Validation Loss
    plt.figure(figsize=(10, 6))
    plt.plot(epochs, history_dict['train_loss'], '-o', label='Training Loss', color='#1f77b4')
    plt.plot(epochs, history_dict['val_loss'], '-o', label='Validation Loss', color='#ff7f0e')
    plt.title('ResNet Training and Validation Loss', fontsize=14)
    plt.xlabel('Epoch', fontsize=12)
    plt.ylabel('Cross-Entropy Loss', fontsize=12)
    plt.grid(True)
    plt.legend(fontsize=11)
    plt.tight_layout()
    plt.savefig('outputs/training_validation_loss.png')
    plt.show()

    #Plot Training and Validation Accuracy
    plt.figure(figsize=(10, 6))
    plt.plot(epochs, history_dict['train_acc'], '-o', label='Training Accuracy', color='#1f77b4')
    plt.plot(epochs, history_dict['val_acc'], '-o', label='Validation Accuracy', color='#ff7f0e')
    plt.title('ResNet Training and Validation Accuracy', fontsize=14)
    plt.xlabel('Epoch', fontsize=12)
    plt.ylabel('Accuracy (%)', fontsize=12)
    plt.grid(True)
    plt.legend(fontsize=11)
    plt.tight_layout()
    plt.savefig('outputs/training_validation_accuracy.png')
    plt.show()

    #Plot Validation Macro F1 Evolution
    plt.figure(figsize=(10, 6))
    plt.plot(epochs, history_dict['val_macro_f1'], '-o', label='Macro F1', color='#1f77b4')
    plt.title('ResNet Validation Macro F1', fontsize=14)
    plt.xlabel('Epoch', fontsize=12)
    plt.ylabel('Macro F1', fontsize=12)
    plt.ylim(0, 1.0)
    plt.grid(True)
    plt.tight_layout()
    plt.savefig('outputs/validation_macro_f1.png')
    plt.show()



one vs rest roc curves

In [ ]:
from sklearn.metrics import precision_recall_fscore_support, roc_curve, auc, accuracy_score, balanced_accuracy_score, precision_score, recall_score, f1_score, jaccard_score, cohen_kappa_score, roc_auc_score
from sklearn.preprocessing import label_binarize
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

def evaluate_advanced_metrics_with_plots(net, val_loader, checkpoints_path, model_name):
    checkpoint_file = checkpoints_path / 'checkpoints' / f'resnet50-{model_name}_BEST.pth'
    net.load_state_dict(torch.load(checkpoint_file, map_location=device))
    net.eval()

    val_preds, val_targets, val_probs = [], [], []

    with torch.no_grad():
        for data in val_loader:
            inputs = data['image'].to(device)
            labels = data['label'].to(device)
            with torch.amp.autocast(device_type='cuda' if torch.cuda.is_available() else 'cpu', enabled=torch.cuda.is_available()):
                outputs = net(inputs)
                probs = F.softmax(outputs, dim=1)
                predicted = outputs.argmax(dim=1)
            val_preds.extend(predicted.cpu().numpy())
            val_targets.extend(labels.cpu().numpy())
            val_probs.extend(probs.cpu().numpy())

    val_targets = np.array(val_targets)
    val_preds = np.array(val_preds)
    val_probs = np.array(val_probs)

    # Target Class Names matching current matrix data shape
    class_names = ['No Damage', 'Minor Damage', 'Major Damage', 'Destroyed']
    n_classes = len(class_names)

    #Compute and Plot One-vs-Rest ROC Curves
    y_one_hot = label_binarize(val_targets, classes=range(n_classes))
    plt.figure(figsize=(10, 6))

    for i in range(n_classes):
        fpr, tpr, _ = roc_curve(y_one_hot[:, i], val_probs[:, i])
        roc_auc = auc(fpr, tpr)
        plt.plot(fpr, tpr, label=f'{class_names[i]}. AUC = {roc_auc:.3f}')

    plt.plot([0, 1], [0, 1], 'k--', label='Random Guess', color='red')
    plt.xlim([-0.05, 1.05])
    plt.ylim([-0.05, 1.05])
    plt.xlabel('False Positive Rate', fontsize=12)
    plt.ylabel('True Positive Rate', fontsize=12)
    plt.title('ResNet One-vs-Rest ROC Curves', fontsize=14)
    plt.legend(loc="lower right", fontsize=11)
    plt.grid(True)
    plt.tight_layout()
    plt.savefig('outputs/one_vs_rest_roc.png')
    plt.show()

    #Compute and Plot Per-Class Classification Metrics
    precision, recall, f1, _ = precision_recall_fscore_support(val_targets, val_preds, labels=range(n_classes), zero_division=0)

    x = np.arange(n_classes)
    width = 0.2

    plt.figure(figsize=(10, 6))
    plt.bar(x - width, precision, width, label='Precision', color='#1f77b4')
    plt.bar(x, recall, width, label='Recall', color='#ff7f0e')
    plt.bar(x + width, f1, width, label='F1 Score', color='#2ca02c')

    plt.title('ResNet Per-Class Classification Metrics', fontsize=14)
    plt.xlabel('Damage Class', fontsize=12)
    plt.ylabel('Score', fontsize=12)
    plt.xticks(x, class_names, fontsize=11)
    plt.ylim(0, 1.05)
    plt.legend(loc='lower right', fontsize=11)
    plt.grid(axis='y')
    plt.tight_layout()
    plt.savefig('outputs/per_class_metrics.png')
    plt.show()

    #Return Dashboard Metrics Table
    loss_criterion = torch.nn.CrossEntropyLoss(weight=criterion.weight)
    test_loss = loss_criterion(torch.tensor(val_probs).to(device), torch.tensor(val_targets).to(device)).item()

    results_data = {
        "Metric": ["Test Cross-Entropy Loss", "Accuracy", "Balanced Accuracy", "Macro Precision", "Macro Recall", "Macro F1", "Weighted F1", "Mean IoU / Jaccard Score", "Quadratic Weighted Kappa", "Macro ROC-AUC, One-vs-Rest"],
        "Value": [test_loss, accuracy_score(val_targets, val_preds), balanced_accuracy_score(val_targets, val_preds), precision_score(val_targets, val_preds, average='macro', zero_division=0), recall_score(val_targets, val_preds, average='macro', zero_division=0), f1_score(val_targets, val_preds, average='macro', zero_division=0), f1_score(val_targets, val_preds, average='weighted', zero_division=0), jaccard_score(val_targets, val_preds, average='macro', zero_division=0), cohen_kappa_score(val_targets, val_preds, weights='quadratic'), roc_auc_score(val_targets, val_probs, multi_class='ovr', average='macro')]
    }
    return pd.DataFrame(results_data)


dataset split distribution bar chart

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

def plot_dataset_splits(train_csv_path, val_csv_path):
    train_df = pd.read_csv(train_csv_path)
    val_df = pd.read_csv(val_csv_path)

    train_counts = train_df['damage_label'].value_counts().sort_index().values
    val_counts = val_df['damage_label'].value_counts().sort_index().values

    class_names = ['No Damage', 'Minor Damage', 'Major Damage', 'Destroyed']
    x = np.arange(len(class_names))
    width = 0.35

    plt.figure(figsize=(10, 6))
    plt.bar(x - width/2, train_counts, width, label='Training', color='#1f77b4')
    plt.bar(x + width/2, val_counts, width, label='Validation', color='#ff7f0e')

    plt.title('Class Distribution by Dataset Split', fontsize=14)
    plt.xlabel('Damage Class', fontsize=12)
    plt.ylabel('Number of Images', fontsize=12)
    plt.xticks(x, class_names, fontsize=11)
    plt.legend(fontsize=11)
    plt.grid(axis='y')
    plt.tight_layout()
    plt.savefig('outputs/dataset_split_distribution.png')
    plt.show()

plot_dataset_splits('/content/xview2-classification-train.csv', '/content/xview2-classification-val.csv')


inference grid visualizer (prediction confidences)

In [ ]:
import torchvision.utils as vutils
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np
import torch

def display_inference_sample_grid(net, val_loader, num_images=6):
    net.eval()
    images_captured, labels_captured, confidence_scores, pred_labels = [], [], [], []
    class_names = ['No Damage', 'Minor Damage', 'Major Damage', 'Destroyed']

    with torch.no_grad():
        for data in val_loader:
            inputs = data['image'].to(device)
            labels = data['label'].to(device)
            outputs = net(inputs)
            probs = F.softmax(outputs, dim=1)

            conf, preds = torch.max(probs, dim=1)

            # Populate sample storage lists until limits are satisfied
            for idx in range(inputs.size(0)):
                if len(images_captured) < num_images:
                    # Reverse normalization transform logic for visual plotting clarity
                    img = inputs[idx].cpu().numpy().transpose((1, 2, 0))
                    mean = np.array([0.485, 0.456, 0.406])
                    std = np.array([0.229, 0.224, 0.225])
                    img = std * img + mean
                    img = np.clip(img, 0, 1)

                    images_captured.append(img)
                    labels_captured.append(labels[idx].item())
                    pred_labels.append(preds[idx].item())
                    confidence_scores.append(conf[idx].item())
            if len(images_captured) >= num_images:
                break

    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    axes = axes.flatten()

    for idx in range(num_images):
        axes[idx].imshow(images_captured[idx])
        axes[idx].axis('off')

        is_correct = labels_captured[idx] == pred_labels[idx]
        title_color = 'green' if is_correct else 'red'

        title_text = (f"True: {class_names[labels_captured[idx]]}\n"
                      f"Predicted: {class_names[pred_labels[idx]]}\n"
                      f"Confidence: {confidence_scores[idx]*100:.1f}%")

        axes[idx].set_title(title_text, color=title_color, fontsize=11, fontweight='bold')

    plt.tight_layout()
    plt.savefig('outputs/model_prediction_sample_grid.png')
    plt.show()

# Generate visual prediction matrix evaluation
display_inference_sample_grid(net, val_loader, num_images=6)


In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import numpy as np
import torch

def generate_confusion_matrix(net, val_loader, checkpoints_path, model_name):
    checkpoint_file = checkpoints_path / 'checkpoints' / f'resnet50-{model_name}_BEST.pth'
    print(f"Loading weights from BEST checkpoint: {checkpoint_file}")
    net.load_state_dict(torch.load(checkpoint_file, map_location=device))

    net.eval()
    all_preds = []
    all_labels = []

    print("Evaluating validation set to collect predictions...")
    with torch.no_grad():
        for data in val_loader:
            inputs = data['image'].to(device)
            labels = data['label'].to(device)
            outputs = net(inputs)
            predicted = outputs.argmax(dim=1)

            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    cm = confusion_matrix(all_labels, all_preds)
    cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

    class_names = ['No Damage', 'Minor Damage', 'Major Damage', 'Destroyed']
    plt.figure(figsize=(8,6))

    labels_display = (np.array([f"{count}\n({percent:.1%})"
                     for count, percent in zip(cm.flatten(), cm_normalized.flatten())])).reshape(4, 4)

    ax = sns.heatmap(cm_normalized, annot=labels_display, fmt="", cmap='viridis',
                     xticklabels=class_names, yticklabels=class_names)
    ax.collections[0].colorbar.ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))

    plt.title(f'xView2 Validation Confusion Matrix (Best Model Peak)', fontsize=14, pad=15)
    plt.ylabel('True Ground Truth Label', fontsize=12)
    plt.xlabel('Model Predicted Label', fontsize=12)
    plt.tight_layout()
    plt.show()


In [ ]:
from sklearn.metrics import confusion_matrix, accuracy_score, balanced_accuracy_score, precision_score, recall_score, f1_score, jaccard_score, cohen_kappa_score, roc_auc_score
from scipy.stats import hmean
import torch.nn.functional as F
import numpy as np
import pandas as pd
import torch

def evaluate_advanced_f1_metrics(net, val_loader, checkpoints_path, model_name):
    checkpoint_file = checkpoints_path / 'checkpoints' / f'resnet50-{model_name}_BEST.pth'
    print(f"Loading weights from BEST checkpoint: {checkpoint_file}")
    net.load_state_dict(torch.load(checkpoint_file, map_location=device))

    net.eval()
    val_preds = []
    val_targets = []
    val_probs = []

    print("Collecting validation predictions for advanced metrics...")
    with torch.no_grad():
        for data in val_loader:
            inputs = data['image'].to(device)
            labels = data['label'].to(device)

            with torch.amp.autocast(device_type='cuda' if torch.cuda.is_available() else 'cpu', enabled=torch.cuda.is_available()):
                outputs = net(inputs)
                probs = F.softmax(outputs, dim=1)
                predicted = outputs.argmax(dim=1)

            val_preds.extend(predicted.cpu().numpy())
            val_targets.extend(labels.cpu().numpy())
            val_probs.extend(probs.cpu().numpy())

    val_targets = np.array(val_targets)
    val_preds = np.array(val_preds)
    val_probs = np.array(val_probs)


    per_class_f1 = f1_score(val_targets, val_preds, average=None)
    macro_f1_val = f1_score(val_targets, val_preds, average='macro')
    per_class_f1_safe = np.clip(per_class_f1, 1e-6, 1.0)
    harmonic_mean_f1 = hmean(per_class_f1_safe)

    print("\n" + "="*40)
    print("Xview2 evaluation metrics (Best Model Peak):")
    print("="*40)
    print(f"Overall Macro F1 Score:         {macro_f1_val:.4f}")
    print(f"Harmonic Mean of F1 Scores:     {harmonic_mean_f1:.4f}")
    print("-"*40)
    class_names = ['No Damage', 'Minor Damage', 'Major Damage', 'Destroyed']
    for idx, name in enumerate(class_names):
        print(f"-> {name.ljust(15)} F1 Score: {per_class_f1[idx]:.4f}")
    print("="*40 + "\n")


    loss_criterion = torch.nn.CrossEntropyLoss(weight=criterion.weight) # Correctly preserves balanced weights
    test_loss = loss_criterion(torch.tensor(val_probs).to(device), torch.tensor(val_targets).to(device)).item()

    accuracy = accuracy_score(val_targets, val_preds)
    balanced_acc = balanced_accuracy_score(val_targets, val_preds)
    macro_precision = precision_score(val_targets, val_preds, average='macro', zero_division=0)
    macro_recall = recall_score(val_targets, val_preds, average='macro', zero_division=0)
    weighted_f1 = f1_score(val_targets, val_preds, average='weighted', zero_division=0)
    mean_iou = jaccard_score(val_targets, val_preds, average='macro', zero_division=0)
    qw_kappa = cohen_kappa_score(val_targets, val_preds, weights='quadratic')
    macro_roc_auc = roc_auc_score(val_targets, val_probs, multi_class='ovr', average='macro')

    results_data = {
        "Metric": [
            "Test Cross-Entropy Loss",
            "Accuracy",
            "Balanced Accuracy",
            "Macro Precision",
            "Macro Recall",
            "Macro F1",
            "Weighted F1",
            "Mean IoU / Jaccard Score",
            "Quadratic Weighted Kappa",
            "Macro ROC-AUC, One-vs-Rest"
        ],
        "Value": [
            test_loss, accuracy, balanced_acc, macro_precision,
            macro_recall, macro_f1_val, weighted_f1, mean_iou, qw_kappa, macro_roc_auc
        ]
    }

    return pd.DataFrame(results_data)


In [ ]:
import torch
import torch.nn as nn
import torchvision
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image
import cv2


class GradCAMResNet(nn.Module):
    def __init__(self, num_classes=4):
        super(GradCAMResNet, self).__init__()
        self.resnet = torchvision.models.get_model('resnet50', weights=None)
        self.resnet.fc = nn.Linear(2048, num_classes)

        self.features = nn.Sequential(
            self.resnet.conv1,
            self.resnet.bn1,
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False),
            self.resnet.layer1,
            self.resnet.layer2,
            self.resnet.layer3,
            self.resnet.layer4
        )
        self.avgpool = self.resnet.avgpool
        self.classifier = self.resnet.fc
        self.gradient = None

    def activations_hook(self, grad):
        self.gradient = grad

    def get_gradient(self):
        return self.gradient

    def get_activations(self, x):
        return self.features(x)

    def forward(self, x):
        x = self.features(x)
        h = x.register_hook(self.activations_hook)
        x = self.avgpool(x)
        x = x.view(x.size(0), -1)
        x = self.classifier(x)
        return x


CHECKPOINT_PATH = 'outputs/checkpoints/resnet50-damage_model_BEST.pth'

gradcam_model = GradCAMResNet(num_classes=4)
gradcam_model.resnet.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=device))
gradcam_model = gradcam_model.to(device)
gradcam_model.eval()

CLASS_NAMES = ['no-damage', 'minor-damage', 'major-damage', 'destroyed']


def run_gradcam(chip_path, true_label=None, save_path='/content/gradcam_map.jpg'):
    raw_image = Image.open(chip_path).convert('RGB')
    img_tensor = composed_val(raw_image).unsqueeze(0).to(device)

    pred = gradcam_model(img_tensor)
    predicted_class = pred.argmax(dim=1).item()

    label_str = f" | True: {CLASS_NAMES[true_label]}" if true_label is not None else ""
    print(f"Predicted: {CLASS_NAMES[predicted_class]}{label_str}")

    gradcam_model.zero_grad()
    pred[:, predicted_class].backward()

    gradients = gradcam_model.get_gradient()
    pooled_gradients = torch.mean(gradients, dim=[0, 2, 3])
    activations = gradcam_model.get_activations(img_tensor).detach()

    num_channels = activations.shape[1]
    for i in range(num_channels):
        activations[:, i, :, :] *= pooled_gradients[i]

    heatmap = torch.mean(activations, dim=1).squeeze()
    heatmap = np.maximum(heatmap.cpu().numpy(), 0)
    heatmap /= (np.max(heatmap) + 1e-8)

    plt.matshow(heatmap)
    plt.title(f"Grad-CAM: predicted {CLASS_NAMES[predicted_class]}")
    plt.show()

    img_cv = cv2.imread(chip_path)
    heatmap_resized = cv2.resize(heatmap, (img_cv.shape[1], img_cv.shape[0]))
    heatmap_uint8 = np.uint8(255 * heatmap_resized)
    heatmap_color = cv2.applyColorMap(heatmap_uint8, cv2.COLORMAP_JET)

    superimposed_img = np.clip(heatmap_color * 0.4 + img_cv, 0, 255).astype('uint8')
    cv2.imwrite(save_path, superimposed_img)
    print(f"Saved overlay to {save_path}")

    return predicted_class


# Trying on chip from validation set
val_df = pd.read_csv('/content/xview2-classification-val.csv')
row = val_df.iloc[0]   # change this index to look at different chips

run_gradcam(row['chip_path'], true_label=int(row['damage_label']))


for damage_class in range(4):
    sample_row = val_df[val_df['damage_label'] == damage_class].iloc[0]
    print(f"\n--- Damage class: {CLASS_NAMES[damage_class]} ---")
    run_gradcam(
        sample_row['chip_path'],
        true_label=damage_class,
        save_path=f'/content/gradcam_class{damage_class}.jpg'
    )